In [2]:
import os
import re
import time
import pandas as pd
from tqdm import tqdm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# ==========================================
# 1. SETUP AMBIENTE E CARICAMENTO DATI
# ==========================================
# Sostituisci "input_dataset.csv" con il percorso reale del tuo file
INPUT_CSV = "df_duplicati.csv"
OUTPUT_CSV_ANONIMO = "dataset_report_anonimizzati_Llama-3.2-3B.csv"

if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(f"Il file '{INPUT_CSV}' non esiste.")

df_input = pd.read_csv(INPUT_CSV)

if "testo_originale" not in df_input.columns or "nome_file" not in df_input.columns:
    raise ValueError("Il DataFrame sorgente deve contenere le colonne 'testo_originale' e 'nome_file'.")

# Checkpoint ottimizzato: lettura singola pre-ciclo $O(1)$ lookup
file_processati = set()
if os.path.exists(OUTPUT_CSV_ANONIMO):
    df_check = pd.read_csv(OUTPUT_CSV_ANONIMO)
    if "nome_file" in df_check.columns:
        file_processati = set(df_check['nome_file'].values)

# ==========================================
# 2. ESTRAZIONE METADATI E PULIZIA TESTO
# ==========================================
def estrai_metadata(nome_file):
    base = nome_file.replace("_KeyResults.txt", "")
    periodo_match = re.search(r'([A-Za-z]{3}_\d{4})_-_([A-Za-z]{3}_\d{4})', base)

    if periodo_match:
        inizio_periodo = periodo_match.group(1).replace('_', ' ')
        fine_periodo = periodo_match.group(2).replace('_', ' ')
        periodo = f"{inizio_periodo} / {fine_periodo}"
    else:
        inizio_periodo = None
        fine_periodo = None
        periodo = base

    paese_match = re.match(r'^(.+?)_[A-Za-z]{3}_\d{4}', base)
    paese = paese_match.group(1).replace('_', ' ') if paese_match else base

    return {
        "paese": paese,
        "periodo": periodo,
        "inizio_periodo": inizio_periodo,
        "fine_periodo": fine_periodo,
        "nome_file": nome_file
    }
df_input

,Unnamed: 0,nome_file,paese,periodo,inizio_periodo,fine_periodo,testo_originale,gruppo,Llama_3_1_8B_anonimo,Llama_3_1_8B_anonimo_promtp_in_context,Llama_3_2_3B_anonimo,Llama_3_1_8B_8bit_anonimo
0,7,El_Salvador_Mar_2025_-_Feb_2026_KeyResults.txt,El Salvador,Mar 2025 / Feb 2026,Mar 2025,Feb 2026,"During the current period (March to May 2025),...",0.0,[SHOCKS AND DRIVERS]: \nThe affected areas are...,[SHOCKS AND DRIVERS]:\nThe affected areas are ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]:\nThe affected areas are ...
1,9,Honduras_Mar_2025_-_Feb_2026_KeyResults.txt,Honduras,Mar 2025 / Feb 2026,Mar 2025,Feb 2026,"During the current period (March to May 2025),...",0.0,[SHOCKS AND DRIVERS]: \nThe affected areas are...,[SHOCKS AND DRIVERS]:\nThe affected areas are ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]:\nThe affected areas are ...
2,23,Guatemala_Mar_2025_-_Feb_2026_KeyResults.txt,Guatemala,Mar 2025 / Feb 2026,Mar 2025,Feb 2026,"During the current period (March to May 2025),...",0.0,[SHOCKS AND DRIVERS]: \nThe affected areas are...,[SHOCKS AND DRIVERS]:\nThe affected areas are ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe affected areas are...
3,31,El_Salvador_Jun_2020_-_Aug_2020_KeyResults.txt,El Salvador,Jun 2020 / Aug 2020,Jun 2020,Aug 2020,"Overview\nFrom June to August 2020, the period...",1.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]:\nThe affected areas expe...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,"[SHOCKS AND DRIVERS]:\nEconomic factors, clima..."
4,35,Honduras_Jun_2020_-_Aug_2020_KeyResults.txt,Honduras,Jun 2020 / Aug 2020,Jun 2020,Aug 2020,"Overview\nFrom June to August 2020, the period...",1.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]:\nThe affected areas expe...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]:\nThe affected areas are ...
5,59,Guatemala_Jun_2020_-_Aug_2020_KeyResults.txt,Guatemala,Jun 2020 / Aug 2020,Jun 2020,Aug 2020,"Overview\nFrom June to August 2020, the period...",1.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]:\nThe affected areas expe...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,"[SHOCKS AND DRIVERS]:\nEconomic factors, clima..."
6,68,El_Salvador_Jun_2022_-_Aug_2022_KeyResults.txt,El Salvador,Jun 2022 / Aug 2022,Jun 2022,Aug 2022,Since several months have passed since the sec...,2.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]:\nThe affected areas expe...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...
7,70,Honduras_Jun_2022_-_Aug_2022_KeyResults.txt,Honduras,Jun 2022 / Aug 2022,Jun 2022,Aug 2022,Since several months have passed since the sec...,2.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]:\nThe affected areas expe...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...
8,74,Guatemala_Jun_2022_-_Aug_2022_KeyResults.txt,Guatemala,Jun 2022 / Aug 2022,Jun 2022,Aug 2022,Since several months have passed since the sec...,2.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]:\nThe affected areas expe...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...
9,162,El_Salvador_Nov_2018_-_Apr_2019_KeyResults.txt,El Salvador,Nov 2018 / Apr 2019,Nov 2018,Apr 2019,The Tri-national Border Federation of Río Lemp...,3.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]:\nThe affected areas expe...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...


In [3]:


# ==========================================
# 3. DOWNLOAD GGUF E INIZIALIZZAZIONE LLAMA.CPP
# ==========================================
print("Recupero dei pesi quantizzati GGUF (Llama-3.1-8B-Instruct, 4-bit)...")
modello_gguf_path = hf_hub_download(
    repo_id="bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
    filename="Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"
)

print("Inizializzazione del motore di inferenza MPS (Metal)...")
llm = Llama(
    model_path=modello_gguf_path,
    n_gpu_layers=-1,
    n_ctx=4096,
    verbose=False
)

# ==========================================
# 4. GESTIONE DEI PROMPT SEMANTICI
# ==========================================
# Regola 5 eliminata come richiesto
esempio_1_input = """During the 2017 post-harvest season, 33% of the population were classified in “Crisis” (IPC Phase 3) and “Emergency” (IPC Phase 4) in the North Eastern Region covering the provinces of Badakhshan, Baghlan, Kunduz and Takhar.

Kohistan, Raghistan, Yawan, Koran o Minjan, Yamgan, Darwaz Bala, Darwaz Payin, Kufab, Shiki, Wakhan, Zebak, Arghanj Khwah, Eshkashim, Shighnan districts of Badakhshan province were the most food insecure districts with 30% to 45% food insecure households, corresponds to 120 293 people in emergency situation (IPC phase 4).

Badakhshan was the most vulnerable province in terms of food insecurity including 5 clusters (14 districts) classified in emergency situation (IPC Phase 4) due to limited food availability in the market, poor access and utilization of food, displacement due to conflicts and remoteness of some districts especially in Darwaz, 3 districts that have no road connection from Afghanistan. Other causes are the road blockage in 12 districts of Badakhshan province for up to 6 months due to heavy snow, mountainous landscape and bad road conditions.

The Kunduz province was partially under control of government last year with high conflict and insecurity rate, which caused displacement of 39000 individuals to centre of Kunduz and neighbouring provinces.

Residents of Takhar have suffered shortages of water for irrigation where bad road conditions also contributed challenging food availability. Insecurity in some districts of this province caused displacement of 8100 individuals.

On-going conflicts, natural disasters, decreased level of water, limited market functionality, remoteness of some areas in this region are the major contributing factors which caused displacement, unemployment, depletion of livelihood assets and led people to engage in irreversible coping strategies."""

esempio_1_output = """[SHOCKS AND DRIVERS]:
The affected areas experienced significant food insecurity due to a combination of economic factors, agricultural challenges, and climate shocks. Limited food availability in the market, poor access and utilization of food, and displacement due to conflicts were major contributing factors. Road blockages for up to 6 months in some districts due to heavy snow and poor road conditions exacerbated the situation. Additionally, ongoing conflicts, natural disasters, and decreased water levels further compounded the issues, leading to displacement, unemployment, and depletion of livelihood assets.

[CURRENT FOOD SECURITY DATA]:
During the post-harvest season, 33% of the population were classified in “Crisis” (IPC Phase 3) and “Emergency” (IPC Phase 4). The most food insecure districts had 30% to 45% of households in emergency situations, corresponding to 120,293 people in IPC Phase 4. Five clusters (14 districts) were classified in emergency situations (IPC Phase 4) due to limited food availability, poor access and utilization of food, and displacement.

[PROJECTED FOOD SECURITY DATA]:
The report does not provide specific projected food security data for future periods. However, the ongoing conflicts, natural disasters, and limited market functionality are expected to continue impacting food security in the affected areas.

[HUMANITARIAN IMPACTS]:
The humanitarian impacts include displacement of 39,000 individuals to the center of one province and neighboring areas due to high conflict and insecurity rates. In another province, 8,100 individuals were displaced due to insecurity in some districts. The overall impact on livelihoods includes unemployment and depletion of livelihood assets, leading people to engage in irreversible coping strategies."""

esempio_2_input = """The Valle and Choluteca region is classified Phase 1 from 4 December 2012 to 31 January 2013.  As from 31 January to 28 February 2013, Phase 1 is maintained for municipalities as Aramecina, Charity, Goascorán, Nacaome, Alliance, Amapala, San Lorenzo, Conception of Mary, Choluteca, El Corpus, El Triunfo, Marcovia, San Marcos Columbus and St. Anne Yusguare.
The classification need to be reviewed for the municipalities of Apacilagua, Duyure, Morolica, Namasigue, Orocuina, Pespire, San Antonio Flores, San Isidro, San Jose, and San Francisco de Langue Coray.
In Amapala, households dedicated to fishing could face a more complicated period due to an increasing pressure on resources in the Gulf and the tension on the jurisdiction of the water control.
Last September, the food consumption level of households in some municipalities was acceptable. Given that current conditions are more favourable than in September, it I is possible to infer this situation quite certainly.
Contributing factors related to food safety are maintained at favourable levels or normal: no adverse weather events, grain prices below the historical average and employment opportunities.
"""
esempio_2_output = """[SHOCKS AND DRIVERS]:
The affected areas have experienced a variety of factors influencing food security. These include stable economic conditions with grain prices below historical averages, favorable weather conditions without adverse events, and the presence of employment opportunities. However, in some coastal areas, households dependent on fishing face challenges due to resource pressure and jurisdictional tensions over water control.

[CURRENT FOOD SECURITY DATA]:
The current food security classification is Phase 1 for the affected areas. This classification is maintained for numerous municipalities, including those where households have shown acceptable food consumption levels. The conditions are more favorable compared to the previous period, supporting the current Phase 1 classification.

[PROJECTED FOOD SECURITY DATA]:
The classification for several municipalities needs to be reviewed, indicating potential changes in food security status. However, the overall conditions remain favorable, with no adverse weather events and stable economic factors, suggesting that the current Phase 1 classification could persist.

[HUMANITARIAN IMPACTS]:
The humanitarian impacts on livelihoods are mixed. While many households are experiencing favorable conditions, those dependent on fishing in certain coastal areas are facing resource pressures and jurisdictional issues, which could affect their livelihoods and food security. There are no significant displacements reported, but the tension over water control could lead to further complications in the future.
"""

# ============================================================
# SISTEMA PROMPT (più breve — gli esempi fanno il lavoro pesante)
# ============================================================
prompt_sistema = (
    "You are an expert humanitarian data analyst. "
    "Your task is to rewrite food insecurity reports into anonymous structured summaries "
    "optimized for semantic similarity analysis.\n\n"
    "Rules:\n"
    "- Remove ALL geographic names (countries, regions, cities) and ALL dates.\n"
    "- Use generic terms: 'the affected areas', 'the population', 'the country'.\n"
    "- Preserve ALL exact numbers, percentages, and IPC classifications.\n"
    "- Structure output in exactly 4 tagged sections as shown in the examples.\n"
    "- No markdown headers or bullet points.\n\n"
    "Learn from these examples:"
)

# ============================================================
# PROMPT STRUTTURA CON ESEMPI
# ============================================================
prompt_struttura = (
    f"{prompt_sistema}\n\n"
    f"--- EXAMPLE 1 ---\n"
    f"INPUT:\n{esempio_1_input}\n\n"
    f"OUTPUT:\n{esempio_1_output}\n\n"
    f"--- EXAMPLE 2 ---\n"
    f"INPUT:\n{esempio_2_input}\n\n"
    f"OUTPUT:\n{esempio_2_output}\n\n"
    f"--- NOW PROCESS THIS REPORT ---\n"
    f"INPUT:\n"
)


# ==========================================
# (Inserisci questo blocco prima del ciclo FOR)
# ==========================================
nome_modello = "Llama_3_1_8B_8bit"
colonna_output = f"{nome_modello}_anonimo_promtp_in_context"

# Inizializza la colonna nel DataFrame se non esiste per evitare KeyError
if colonna_output not in df_input.columns:
    df_input[colonna_output] = pd.NA

print(f"\nTrovati {len(df_input)} report nel DataFrame da elaborare...\n")

# ==========================================
# 5. ELABORAZIONE DEL BATCH DAL DATAFRAME
# ==========================================
for index, row in tqdm(df_input.iterrows(), total=len(df_input), desc="Elaborazione report"):

    # Lookup vettoriale per evitare inferenze ridondanti:
    # Se la colonna contiene già un valore per questa riga, salta.
    if pd.notna(row.get(colonna_output)):
        continue

    nome_file = row['nome_file']
    testo_originale = str(row['testo_originale'])
    testo_pulito = re.sub(r'^.*?={20,}\n*', '', testo_originale, flags=re.DOTALL).strip()

    if not testo_pulito or testo_pulito.lower() == 'nan':
        print(f" -> Avviso: il testo per {nome_file} risulta vuoto dopo la pulizia.")
        continue

    messages = [
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
    ]

    try:
        outputs = llm.create_chat_completion(
            messages=messages,
            max_tokens=700,
            temperature=0.1
        )

        scheda_anonima = outputs["choices"][0]["message"]["content"].strip()
        scheda_anonima = re.sub(r'###.*?\n', '', scheda_anonima).strip()

        # Inserimento atomico del risultato nel DataFrame in RAM
        df_input.loc[index, colonna_output] = scheda_anonima

    except Exception as e:
        print(f"\nErrore di esecuzione sul file {nome_file}: {e}")
        time.sleep(2)



Recupero dei pesi quantizzati GGUF (Llama-3.1-8B-Instruct, 4-bit)...


Inizializzazione del motore di inferenza MPS (Metal)...

Trovati 35 report nel DataFrame da elaborare...



Elaborazione report: 100%|██████████| 35/35 [07:09<00:00, 12.27s/it]


In [4]:
# ==========================================
# 6. ESPORTAZIONE FINALE (OBBLIGATORIA)
# ==========================================
# Sovrascrive il file originale (o ne crea uno nuovo) con la colonna aggiunta
df_input.to_csv(INPUT_CSV, index=False, encoding='utf-8')
print(f"\nPipeline completata. DataFrame salvato in: '{INPUT_CSV}' con la nuova colonna '{colonna_output}'.")


Pipeline completata. DataFrame salvato in: 'df_duplicati.csv' con la nuova colonna 'Llama_3_1_8B_8bit_anonimo_promtp_in_context'.
